In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sts
from statsmodels.formula.api import ols
import statsmodels.api as sm

In [38]:
data = pd.read_csv('../data/final.csv')
data['log_scored_by'] = np.log10(data['scored_by'])
data['sqrt_chapters'] = np.power(data['chapters'], 0.5)
columns = ['A', 'log_scored_by', 'volumes', 'sqrt_chapters',  'has Romance', 'has Comedy', 'has Hentai', 'has Fantasy',
       'has Boys Love', 'has School', 'has Historical', 'has Harem',
       'has Psychological', 'has Isekai']

data = data.sample(frac=1)
print(len(data))
tests = data.head(20)
print(len(tests))
data = data.tail(- len(tests))
print(len(data))

180
20
160


In [39]:
def load_YX(data, columns):
  X = data[columns[1:]].to_numpy()

  X = np.hstack((np.ones((X.shape[0], 1)), X))
  Y = data[['score']].to_numpy()
  return Y, X

Y, X = load_YX(data, columns)

In [40]:
def solve_multi_linear_model(Y, X):
  B = np.linalg.inv(X.T @ X) @ X.T @ Y
  e = Y - X @ B
  return B, e

B, e = solve_multi_linear_model(Y, X)

In [41]:
def get_corr(columns):
  return data[['score'] + columns[1:]].corr().round(2)

get_corr(columns)

,score,log_scored_by,volumes,sqrt_chapters,has Romance,has Comedy,has Hentai,has Fantasy,has Boys Love,has School,has Historical,has Harem,has Psychological,has Isekai
score,1.00,0.46,0.28,0.39,0.02,0.01,-0.10,0.01,-0.14,0.06,0.01,-0.03,-0.00,0.06
log_scored_by,0.46,1.00,0.36,0.36,0.06,0.17,-0.13,-0.02,-0.15,0.14,-0.05,-0.05,0.04,-0.05
volumes,0.28,0.36,1.00,0.88,0.20,0.19,-0.11,0.22,-0.22,0.17,-0.01,0.13,-0.09,-0.03
sqrt_chapters,0.39,0.36,0.88,1.00,0.13,0.20,-0.08,0.22,-0.30,0.20,0.03,0.12,-0.07,0.08
has Romance,0.02,0.06,0.20,0.13,1.00,0.23,-0.14,-0.16,-0.33,0.08,0.06,0.03,0.15,-0.05
has Comedy,0.01,0.17,0.19,0.20,0.23,1.00,-0.12,0.07,-0.14,0.05,-0.17,-0.11,-0.01,-0.05
has Hentai,-0.10,-0.13,-0.11,-0.08,-0.14,-0.12,1.00,-0.04,-0.10,-0.08,-0.06,0.34,-0.05,-0.02
has Fantasy,0.01,-0.02,0.22,0.22,-0.16,0.07,-0.04,1.00,-0.28,-0.07,0.21,0.06,-0.13,0.14
has Boys Love,-0.14,-0.15,-0.22,-0.30,-0.33,-0.14,-0.10,-0.28,1.00,-0.04,-0.09,-0.10,-0.12,-0.04
has School,0.06,0.14,0.17,0.20,0.08,0.05,-0.08,-0.07,-0.04,1.00,-0.11,0.14,-0.09,-0.03


In [42]:
def standart_coef(Y, X, B):
  sigma_y = np.sqrt(np.sum((Y - np.mean(Y)) ** 2) / Y.shape[0])
  sigma_x = np.sqrt(np.array([np.sum((X - np.mean(X, axis=0)) ** 2, axis=0)]) / X.shape[0]) 
  return (B[1:] * sigma_y) / sigma_x.T[1:]

def print_cool_B(B):
  ret = f"y = {B[0][0]:.4f} "
  for i in range(1, B.shape[0]):
    ret += f"+ {B[i][0]:.4f} x_{i} "
  return ret

def print_cool_B_norm(B_norm):
  ret = f"t_y = {B_norm[0][0]:.2f} t_x_1 "
  for i in range(1, B_norm.shape[0]):
    ret += f"+ {B_norm[i][0]:.2f} t_x_{i + 1} "
  return ret

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))


y = 5.8192 + 0.4306 x_1 + -0.0460 x_2 + 0.1017 x_3 + 0.0153 x_4 + -0.1592 x_5 + -0.1863 x_6 + -0.0510 x_7 + -0.0244 x_8 + -0.1052 x_9 + -0.0621 x_10 + -0.0700 x_11 + -0.0652 x_12 + 0.0785 x_13 
t_y = 0.48 t_x_1 + -0.01 t_x_2 + 0.02 t_x_3 + 0.02 t_x_4 + -0.21 t_x_5 + -0.58 t_x_6 + -0.07 t_x_7 + -0.04 t_x_8 + -0.18 t_x_9 + -0.14 t_x_10 + -0.24 t_x_11 + -0.18 t_x_12 + 0.59 t_x_13 


In [43]:
def pretty_table(data):
  p = []
  for line in data:
    while len(p) < len(line):
      p.append(0)
    for i in range(len(line)):
      p[i] = max(p[i], len(line[i]))
  
  def print_line(line):
    def add_spaces(s, c):
      return (" " * c) + s
    ret = add_spaces(line[0], p[0] - len(line[0]))
    for i in range(1, len(line)):
      ret += " │ " + add_spaces(line[i], p[i] - len(line[i]))
    for i in range(len(line), len(p)):
      ret += " │ " + p[i] * " "
    return ret

  line_sep = "─" * p[0]
  for i in range(1, len(p)):
    line_sep += "─┼─" + "─" * p[i]

  ret = ""
  ret += print_line(data[0]) + "\n" + line_sep + '\n'
  for i in range(1, len(data)):
    ret += print_line(data[i]) + "\n"
  return ret


In [44]:
def t_Student_criterion(Y, X, B, e, columns, is_print = True, alpha = 0.05):
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  S_2 = e.T @ e / (dfd) # стандартная ошибка в квадрате
  Cov_B = S_2 * np.linalg.inv(X.T @ X)

  SE = np.sqrt(S_2 * np.array([[Cov_B[i][i]] for i in range(Cov_B.shape[1]) ]))

  t_fact = B / SE

  t_table = sts.t.ppf(1 - alpha / 2, dfd)


  table = [["Название критерия", "B", "t_fact", "B!=0"]]

  ret = []

  for i in range(len(columns)):
    table.append([
      columns[i],
      f"{B[i][0]:.5f}",
      f"{t_fact[i][0]:.3f}",
      "+" if np.abs(t_fact[i][0]) > t_table else "-"
    ])
    ret.append(np.abs(t_fact[i][0]) > t_table)
  
  if is_print:
    print(f"t_table = {t_table:.2f}")
    print()
    print(pretty_table(table))
  return ret, t_fact

t_Student_criterion(Y, X, B, e, columns)
pass

t_table = 1.98

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.81918 │ 46.947 │    +
    log_scored_by │  0.43064 │ 10.026 │    +
          volumes │ -0.04602 │ -4.713 │    +
    sqrt_chapters │  0.10171 │  7.659 │    +
      has Romance │  0.01533 │  0.284 │    -
       has Comedy │ -0.15919 │ -3.082 │    +
       has Hentai │ -0.18625 │ -1.517 │    -
      has Fantasy │ -0.05104 │ -0.903 │    -
    has Boys Love │ -0.02440 │ -0.398 │    -
       has School │ -0.10518 │ -1.653 │    -
   has Historical │ -0.06206 │ -0.728 │    -
        has Harem │ -0.07003 │ -0.531 │    -
has Psychological │ -0.06515 │ -0.657 │    -
       has Isekai │  0.07846 │  0.285 │    -



In [45]:
def F_criterion(Y, X, B, e, alpha = 0.05):
  SS_all = np.sum((Y - np.mean(Y)) ** 2)
  SS_R = np.sum((X @ B - np.mean(Y)) ** 2)
  SS_e = np.sum(e ** 2)
  R_2 = SS_R / SS_all
  r = np.sqrt(R_2)

  R_2_norm = 1 - (1 - R_2) * (X.shape[0] - 1) / (X.shape[0] - X.shape[1])
  r_norm =  np.sqrt(R_2_norm)

  dfn = X.shape[1] - 1           # Степени свободы числителя (межгрупповые)
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  MS_R = SS_R / (dfn)
  MS_e = SS_e / (dfd)

  F_fact = MS_R / MS_e

  F_table = sts.f.ppf(1 - alpha, dfn, dfd)


  print(f"SS_all = {SS_all:.3f}, SS_R = {SS_R:.3f}, SS_e = {SS_e:.3f} ")
  print(f"Проверка: SS_R + SS_e = {SS_R + SS_e:.3f}")
  print(f"общий коэффициент детерминации: {R_2:.3f}")
  print(f"общий коэффициент корреляции: {r:.3f}")
  print(f"скорректированный коэффициент детерминации: {R_2_norm:.3f}")
  print(f"скорректированный коэффициент корреляции: {r_norm:.3f}")
  
  print(f"""тестнота связи: {
    "не наблюдается" if r_norm < 0.1 else 
    "слабая" if r_norm < 0.3 else
    "умеренная" if r_norm < 0.5 else
    "заметная" if r_norm < 0.7 else
    "высокая" if r_norm < 0.9 else
    "весьма высокая"
  }""")
  print()
  table = [
    ["", "df", "SS", "MS", "F", "значимость F"],
    ["Регрессия", f"{dfn}", f"{SS_R:.3f}", f"{MS_R:.3f}", f"{F_fact:.3f}", f"{F_table:.3f}"],
    ["Остаток", f"{dfd}", f"{SS_e:.3f}", f"{MS_e:.3f}"],
    ["Итого", f"{dfn + dfd}", f"{SS_all:.3f}"]
  ]
  print(pretty_table(table))
  print()
  print("Линейность наблюдается" if F_fact > F_table else "Нет оснований предпологать линейность")

F_criterion(Y, X, B, e)

SS_all = 56.276, SS_R = 18.138, SS_e = 38.138 
Проверка: SS_R + SS_e = 56.276
общий коэффициент детерминации: 0.322
общий коэффициент корреляции: 0.568
скорректированный коэффициент детерминации: 0.262
скорректированный коэффициент корреляции: 0.512
тестнота связи: заметная

          │  df │     SS │    MS │     F │ значимость F
──────────┼─────┼────────┼───────┼───────┼─────────────
Регрессия │  13 │ 18.138 │ 1.395 │ 5.341 │        1.788
  Остаток │ 146 │ 38.138 │ 0.261 │       │             
    Итого │ 159 │ 56.276 │       │       │             


Линейность наблюдается


In [46]:
columns_clear = columns.copy()

while len(columns_clear) > 0:
  Y, X = load_YX(data, columns_clear)
  B, e = solve_multi_linear_model(Y, X)
  is_goods, t_fact = t_Student_criterion(Y, X, B, e, columns_clear, False)
  is_out = True
  for is_good in is_goods:
    is_out = is_good and is_out
  if is_out:
    break
  i_del = np.argmin(np.abs(t_fact.T[0]))
  columns_clear.pop(i_del)

Y, X = load_YX(data, columns_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_clear)
F_criterion(Y, X, B, e)


y = 5.7611 + 0.4388 x_1 + -0.0448 x_2 + 0.0977 x_3 + -0.1394 x_4 
t_y = 0.49 t_x_1 + -0.01 t_x_2 + 0.02 t_x_3 + -0.19 t_x_4 

t_table = 1.98

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.76115 │ 55.420 │    +
    log_scored_by │  0.43876 │ 11.005 │    +
          volumes │ -0.04480 │ -5.260 │    +
    sqrt_chapters │  0.09765 │  8.459 │    +
       has Comedy │ -0.13937 │ -3.030 │    +

SS_all = 56.276, SS_R = 17.624, SS_e = 38.652 
Проверка: SS_R + SS_e = 56.276
общий коэффициент детерминации: 0.313
общий коэффициент корреляции: 0.560
скорректированный коэффициент детерминации: 0.295
скорректированный коэффициент корреляции: 0.544
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   4 │ 17.624 │ 4.406 │ 17.668 │        2.430
  Остаток │ 155 │ 38.652 │ 0.249 │        │             
    Итого │ 159 │ 56.276 │       │    

In [47]:
corr = get_corr(columns_clear)
print(corr)

               score  log_scored_by  volumes  sqrt_chapters  has Comedy
score           1.00           0.46     0.28           0.39        0.01
log_scored_by   0.46           1.00     0.36           0.36        0.17
volumes         0.28           0.36     1.00           0.88        0.19
sqrt_chapters   0.39           0.36     0.88           1.00        0.20
has Comedy      0.01           0.17     0.19           0.20        1.00


In [48]:
columns_very_clear = columns_clear.copy()
try:
  columns_very_clear.remove("volumes")
  pass
except:
  pass

Y, X = load_YX(data, columns_very_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_very_clear)
F_criterion(Y, X, B, e)

y = 5.8756 + 0.4204 x_1 + 0.0454 x_2 + -0.1447 x_3 
t_y = 0.47 t_x_1 + 0.01 t_x_2 + -0.20 t_x_3 

t_table = 1.98

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.87564 │ 55.697 │    +
    log_scored_by │  0.42042 │ 10.200 │    +
    sqrt_chapters │  0.04536 │  7.447 │    +
       has Comedy │ -0.14472 │ -3.032 │    +

SS_all = 56.276, SS_R = 15.903, SS_e = 40.373 
Проверка: SS_R + SS_e = 56.276
общий коэффициент детерминации: 0.283
общий коэффициент корреляции: 0.532
скорректированный коэффициент детерминации: 0.269
скорректированный коэффициент корреляции: 0.518
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   3 │ 15.903 │ 5.301 │ 20.483 │        2.663
  Остаток │ 156 │ 40.373 │ 0.259 │        │             
    Итого │ 159 │ 56.276 │       │        │             


Линейность наблюдается


In [49]:
corr = get_corr(columns_very_clear)
print(corr)

               score  log_scored_by  sqrt_chapters  has Comedy
score           1.00           0.46           0.39        0.01
log_scored_by   0.46           1.00           0.36        0.17
sqrt_chapters   0.39           0.36           1.00        0.20
has Comedy      0.01           0.17           0.20        1.00


In [50]:
# Прогноз
def test_predict(columns):
  X, Y = load_YX(data, columns)
  B, e = solve_multi_linear_model(X, Y)
  test_Y, test_X = load_YX(tests, columns)
  
  test_Y_pred = test_X @ B
  test_e = test_Y - test_Y_pred
  test_e_2 = test_e ** 2
  sum_e_2 = np.sum(test_e_2)
  sum_e = np.sqrt(sum_e_2 / len(tests))

  table = [["Реальная оценка", "Предсказаная оценка", "Ошибка", "Квадрат ошибки"]]
  for i in range(test_Y.shape[0]):
    table.append([
      f"{test_Y[i][0]:.2f}",
      f"{test_Y_pred[i][0]:.2f}",
      f"{test_e[i][0]:.2f}",
      f"{test_e_2[i][0]:.2f}",
    ])
  table.append([
    "", 
    "Сумма", "--",
    f"{sum_e_2:.2f}",
  ])
  table.append([
    "", 
    "Нормирования ошибка", "--",
    f"!! {sum_e:.2f} !!",
  ])
  print(pretty_table(table))
  return sum_e

In [51]:
e_very_clear = test_predict(columns_very_clear)

Реальная оценка │ Предсказаная оценка │ Ошибка │ Квадрат ошибки
────────────────┼─────────────────────┼────────┼───────────────
           7.23 │                7.26 │  -0.03 │           0.00
           7.67 │                7.75 │  -0.08 │           0.01
           7.15 │                7.03 │   0.12 │           0.02
           6.87 │                6.94 │  -0.07 │           0.00
           7.40 │                7.27 │   0.13 │           0.02
           7.64 │                7.30 │   0.34 │           0.11
           6.86 │                6.75 │   0.11 │           0.01
           6.80 │                7.18 │  -0.38 │           0.15
           7.95 │                7.12 │   0.83 │           0.70
           6.33 │                6.92 │  -0.59 │           0.34
           6.96 │                6.64 │   0.32 │           0.10
           7.63 │                6.97 │   0.66 │           0.43
           6.86 │                7.21 │  -0.35 │           0.12
           6.83 │                7.02 │ 

In [52]:
e_clear = test_predict(columns_clear)

Реальная оценка │ Предсказаная оценка │ Ошибка │ Квадрат ошибки
────────────────┼─────────────────────┼────────┼───────────────
           7.23 │                7.40 │  -0.17 │           0.03
           7.67 │                7.95 │  -0.28 │           0.08
           7.15 │                6.97 │   0.18 │           0.03
           6.87 │                6.93 │  -0.06 │           0.00
           7.40 │                7.11 │   0.29 │           0.08
           7.64 │                7.34 │   0.30 │           0.09
           6.86 │                6.82 │   0.04 │           0.00
           6.80 │                7.50 │  -0.70 │           0.49
           7.95 │                7.09 │   0.86 │           0.73
           6.33 │                6.87 │  -0.54 │           0.30
           6.96 │                6.57 │   0.39 │           0.15
           7.63 │                7.04 │   0.59 │           0.34
           6.86 │                7.19 │  -0.33 │           0.11
           6.83 │                7.08 │ 

In [53]:
e_standart = test_predict(columns)

Реальная оценка │ Предсказаная оценка │ Ошибка │ Квадрат ошибки
────────────────┼─────────────────────┼────────┼───────────────
           7.23 │                7.39 │  -0.16 │           0.03
           7.67 │                8.03 │  -0.36 │           0.13
           7.15 │                6.98 │   0.17 │           0.03
           6.87 │                6.99 │  -0.12 │           0.01
           7.40 │                7.12 │   0.28 │           0.08
           7.64 │                7.28 │   0.36 │           0.13
           6.86 │                6.80 │   0.06 │           0.00
           6.80 │                7.57 │  -0.77 │           0.60
           7.95 │                7.09 │   0.86 │           0.75
           6.33 │                6.89 │  -0.56 │           0.32
           6.96 │                6.57 │   0.39 │           0.15
           7.63 │                7.06 │   0.57 │           0.32
           6.86 │                7.22 │  -0.36 │           0.13
           6.83 │                7.12 │ 

In [54]:
best = ""

if e_very_clear < e_clear and e_very_clear < e_standart:
  best = "хорошо отчищеных выбранных параметров"
elif e_clear < e_standart:
  best = "отчищеных выбранных параметров"
else:
  best = "всех выбранных параметров"

print(f"Лучшие предсказания у {best}")

Лучшие предсказания у хорошо отчищеных выбранных параметров
